In [34]:
#각종 함수 선언 셀, process_dataframe, generate_pivot_statistics, expand_window
import pandas as pd
import numpy as np
import gc
from pandas.api.types import is_categorical_dtype
import re
from pathlib import Path
from datetime import datetime

def process_dataframe(result_df: pd.DataFrame) -> pd.DataFrame:
    # ---- 기본 방어: Series가 들어오면 DF로 ----
    if isinstance(result_df, pd.Series):
        result_df = result_df.reset_index(name=result_df.name or "ID")

    # ---- 인덱스가 의미있는 경우만 풀기 (MultiIndex/Named index) ----
    if not isinstance(result_df.index, pd.RangeIndex) or result_df.index.name is not None:
        result_df = result_df.reset_index()

    # V_label 덧붙이기
    if ("V_label" in result_df.columns) and ("V_label_0" not in result_df.columns):
        data = {
            "V_label": ["VV", "VA", "VCN", "VCP", "XSV", "XSA", "VX"],
            "V_label_0": ["VV", "VA", "VA", "VP", "VV", "VA", "VX"]
        }
        df_v_label = pd.DataFrame(data)
        result_df = pd.merge(result_df, df_v_label, on="V_label", how="left")

    # EN 정보 덧붙이기
    if ("EN_No" in result_df.columns) and ("EN_No_0" not in result_df.columns):
        en = pd.to_numeric(result_df["EN_No"], errors="coerce")
        
        scaled = (en * 10).round().astype("Int64")   # ✅ float 오차 제거용
        result_df["EN_No_0"] = (scaled // 10).astype("Int64")
        result_df["EN_No_1"] = (scaled % 10).astype("Int64")  # 0~9 정수

    # EP정보 덧붙이기
    if 'EP_form' in result_df.columns and 'EP_T' not in result_df.columns:
        # precompile regexes with NON-capturing groups (no warnings)
        re_tt = re.compile(r'(?:었었|았었)')
        re_t  = re.compile(r'(?:었|았)')
        re_m  = re.compile(r'(?:겠|겄)')

        # --- EP_form & flags ---
        if 'EP_form' in result_df.columns:
            s = result_df['EP_form'].astype('string').str.replace(' + ', '', regex=False)
            result_df['EP_form'] = s
            tt = s.str.contains(re_tt, na=False)
            result_df['EP_TT'] = tt
            result_df['EP_T']  = (~tt) & s.str.contains(re_t, na=False)
            result_df['EP_M']  = s.str.contains(re_m, na=False)

    # V_No 정보 덧붙이기 (V_label이 없는 경우)
    if ("V_No" in result_df.columns) and ("V_label" not in result_df.columns):
        v = pd.to_numeric(result_df["V_No"], errors="coerce")

        def assign_v_label(v_no):
            if pd.isna(v_no):
                return ""
            if 0 <= v_no < 1000:
                return "VX"
            elif 1000 <= v_no < 2000:
                return "VC"
            elif 2000 <= v_no < 3000:
                return "VA"
            elif 3000 <= v_no < 5000:
                return "VV"
            else:
                return ""

        result_df["V_label"] = v.apply(assign_v_label)

    return result_df

#개선된 그룹바이 함수
def pivot_statistics(
    df: pd.DataFrame,
    index_columns,
    filter_fn=None,
):
    """
    단일 DF에서 index_columns 조합별 빈도(size)를 계산해 Series로 반환.
    - dropna=False: 키 NA 포함
    - observed=True: category 조합 폭발 방지
    - sort=False: 속도
    """
    if filter_fn is not None:
        df = filter_fn(df)

    try:
        s = (
            df.groupby(index_columns, dropna=False, observed=True, sort=False)
              .size()
              .rename("ID")
        )
    except TypeError:
        # dropna 파라미터 호환 이슈 대비 (구버전/특정 상황)
        SENTINEL = "__<NA>__"
        sub = df[index_columns].copy()
        for c in index_columns:
            sub[c] = sub[c].where(sub[c].notna(), SENTINEL)
        s = sub.value_counts(sort=False).rename("ID")
        s.index = pd.MultiIndex.from_tuples(
            [tuple(np.nan if x == SENTINEL else x for x in tup) for tup in s.index],
            names=index_columns
        )
    return s

# Null값 등으로 인해서 제외되는 값이 있는지 확인하는 함수
def debug_drop_reason(df, index_columns):
    n_all = len(df)
    n_key_na = df[index_columns].isna().any(axis=1).sum()     # 키에 NA 있는 행 수
    n_id_na = df['ID'].isna().sum() if 'ID' in df.columns else 0

    # pivot_table식 count가 실제로 세는 개수(대략적인 감산식)
    approx_pivot_count = df.loc[~df[index_columns].isna().any(axis=1), 'ID'].notna().sum()

    # 권장 경로: groupby + size (키 NA 포함)
    gb_size = df.groupby(index_columns, dropna=False, observed=True).size().sum()

    return {
        "len(df)": int(n_all),
        "키에 NA 있는 행": int(n_key_na),
        "ID가 NA인 행": int(n_id_na),
        "pivot_table(count) 합": int(approx_pivot_count),
        "groupby(size, dropna=False) 합": int(gb_size),
    }

def pivot_statistics_to_df(
    df: pd.DataFrame,
    index_columns,
    filter_fn=None,
    process_fn=None,
) -> pd.DataFrame:
    s = pivot_statistics(df, index_columns, filter_fn=filter_fn)
    out = s.reset_index()  # columns: index_columns + ['ID']

    # ✅ 후처리는 여기서 DF에 직접 적용 (결과 작아진 뒤)
    if process_fn is not None:
        out = process_fn(out)

    return out


In [14]:
### 읽어오기
CSV_PATH = r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver9.csv"
df = pd.read_csv(CSV_PATH)

In [15]:
df.head(100)

,ID,file_id,category,sen_id,word_id2,form/label,N_form,N_label,V_form,V_label,...,vx0_No,vx0_form,vx0_order,vx_len,V_word_id,f_word_id,EP_TT,EP_T,EP_M,sent_end_V
0,AA0001.00005.0001,AA0001,보도·해설,AA0001.00005,1,1993/SN + //SP + 06/SN + //SP + 08/SN,NaN,NaN,NaN,NaN,...,-1.0,NaN,-1.0,0,-1.0,-1.0,False,False,False,False
1,AA0001.00006.0001,AA0001,보도·해설,AA0001.00006,1,19/SN,NaN,NaN,NaN,NaN,...,-1.0,NaN,-1.0,0,-1.0,-1.0,False,False,False,False
2,AA0001.00007.0001,AA0001,보도·해설,AA0001.00007,1,엠마누엘/NNP,NaN,NaN,NaN,NaN,...,-1.0,NaN,-1.0,0,-1.0,-1.0,False,False,False,False
3,AA0001.00007.0002,AA0001,보도·해설,AA0001.00007,2,웅가/NNP + 로/JKB,웅가,NNP,NaN,NaN,...,-1.0,NaN,-1.0,0,-1.0,-1.0,False,False,False,False
4,AA0001.00007.0003,AA0001,보도·해설,AA0001.00007,3,//SP,NaN,NaN,NaN,NaN,...,-1.0,NaN,-1.0,0,-1.0,-1.0,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,AA0001.00014.0002,AA0001,보도·해설,AA0001.00014,2,욕실/NNG + 에서/JKB,욕실,NNG,NaN,NaN,...,-1.0,NaN,-1.0,0,-1.0,-1.0,False,False,False,False
96,AA0001.00014.0003,AA0001,보도·해설,AA0001.00014,3,1/SN + 시간/NNB,NaN,NaN,NaN,NaN,...,-1.0,NaN,-1.0,0,-1.0,-1.0,False,False,False,False
97,AA0001.00014.0004,AA0001,보도·해설,AA0001.00014,4,반/NNG,NaN,NaN,NaN,NaN,...,-1.0,NaN,-1.0,0,-1.0,-1.0,False,False,False,False
98,AA0001.00014.0005,AA0001,보도·해설,AA0001.00014,5,이상/NNG + 을/JKO,이상,NNG,NaN,NaN,...,-1.0,NaN,-1.0,0,-1.0,-1.0,False,False,False,False


In [16]:
df.columns.all

<bound method Index.all of Index(['ID', 'file_id', 'category', 'sen_id', 'word_id2', 'form/label',
       'N_form', 'N_label', 'V_form', 'V_label', 'EP_form', 'EP_label',
       'EN_form', 'EN_label', 'J_form', 'J_label', 'EN_No', 'Next_VX_No',
       'Next_VX_No_form', 'vx0_No', 'vx0_form', 'vx0_order', 'vx_len',
       'V_word_id', 'f_word_id', 'EP_TT', 'EP_T', 'EP_M', 'sent_end_V'],
      dtype='object')>

In [37]:
OUT_DIR = r"..\csv\통계용"
PREFIX = "EP_VX_E_Category_FILE"
index_columns = ['V_label', 'category','file_id','EP_form', 'EN_form', 'EN_No', 'EN_label', 'vx0_No','vx0_order', 'vx_len', 'sent_end_V']
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")

#에러 확인
if True:
    print(debug_drop_reason(df, index_columns))

#실행
result_df = pivot_statistics_to_df(
    df =df,
    index_columns=index_columns,
    process_fn=process_dataframe,
    #filter_fn=filter
)
print("result_df length:", len(result_df))

# 결과 DataFrame 저장

save_folder = Path(OUT_DIR) 
output_file_name = save_folder / f"{PREFIX}_{timestamp}.csv"
output_file_name.parent.mkdir(parents=True, exist_ok=True)  # 폴더 생성

result_df.to_csv(output_file_name, index=True, encoding='utf-8-sig')
print(f"*** 저장 완료: {output_file_name} ({len(result_df):,}행) ***")

{'len(df)': 12973652, '키에 NA 있는 행': 12336207, 'ID가 NA인 행': 0, 'pivot_table(count) 합': 637445, 'groupby(size, dropna=False) 합': 12973652}
result_df length: 489227
*** 저장 완료: ..\csv\통계용\EP_VX_E_Category_FILE_2026-01-09_21-32.csv (489,227행) ***


In [27]:
result_df.columns

Index(['V_form', 'V_label', 'category', 'EP_T', 'EN_form', 'EN_No', 'EN_label',
       'vx0_No', 'vx0_order', 'vx_len', 'sent_end_V', 'ID', 'V_label_0',
       'EN_No_0', 'EN_No_1'],
      dtype='object')